# Turnkey runner — train & compare all three models

One-click pipeline on **Google Colab (GPU)**: download → preprocess →
train *image / text / fusion* → evaluate each on test → assemble the
**3-way comparison** (the project's headline result).

Expected wall-clock on a free Colab GPU: roughly 20–40 min total for the
~20k-row subset (image/fusion dominate; the EfficientNet backbone is frozen).

## 0. Setup — clone the repo & install deps

Runs on **Google Colab**: set the runtime to **GPU** first
(`Runtime → Change runtime type → T4 GPU`). This cell is safe to re-run.

In [ ]:
import os
if not os.path.isdir('/content/DeepLearningProject'):
    !git clone -b claude/build-multimodal-classifier https://github.com/yigitdagidir/DeepLearningProject.git /content/DeepLearningProject
%cd /content/DeepLearningProject
!git pull --ff-only   # pick up any new commits on re-run
!pip -q install -r requirements.txt

## 0b. Kaggle credentials (needed to download the dataset)

The Kaggle *Fashion Product Images (small)* dataset needs a (free) Kaggle
account + API token: **kaggle.com → your avatar → Settings → API →
Create New API Token** (downloads `kaggle.json`). Run the cell below and
upload that file. *(Alternative: uncomment Option B and paste your creds.)*

In [ ]:
# Option A — upload kaggle.json:
from google.colab import files
files.upload()  # choose your kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Option B — set credentials directly (instead of Option A):
# import os
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY']      = 'your_api_key'

In [ ]:
import tensorflow as tf
from src import config
config.set_seeds()
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
print('seed', config.SEED, '| epochs', config.EPOCHS, '| batch', config.BATCH_SIZE)

## 1. Data: download + stratified splits

In [ ]:
from src.data.download import download
from src.data.preprocess import preprocess
download()
preprocess()

In [ ]:
from src.data.dataset import get_text_vectorizer
vec = get_text_vectorizer()  # adapted on TRAIN only, then persisted
print('vocab size:', len(vec.get_vocabulary()))

## 2. Train the three models
Each call saves model + metrics + training curves under `artifacts/<model>/`.

In [ ]:
from src.train import train
train('image')

In [ ]:
train('text')

In [ ]:
train('fusion')

## 3. Evaluate each on the held-out test set
Writes `test_metrics.json` + a confusion matrix per model.

In [ ]:
from src.evaluate import evaluate
for m in ('image', 'text', 'fusion'):
    evaluate(m)

## 4. The 3-way comparison (headline result)

In [ ]:
from src.compare import compare
df = compare()
df

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage(str(config.ARTIFACTS_DIR / 'comparison.png')))
for m in ('image', 'text', 'fusion'):
    print(m)
    display(IPImage(str(config.ARTIFACTS_DIR / m / 'confusion_matrix.png')))

## 5. Copy the result into the report/slides
Paste the printed table into `docs/REPORT.md` §8 and `docs/PRESENTATION.md`.

In [ ]:
print(open(config.ARTIFACTS_DIR / 'comparison.md').read())

## 6. (Optional, Phase 4) Lightweight hyperparameter search on fusion
A handful of manual runs from `config.HP_GRID` — no AutoML.

In [ ]:
results = []
for i, hp in enumerate(config.HP_GRID):
    name = f"fusion_hp{i}"
    s = train('fusion', learning_rate=hp['learning_rate'], dropout=hp['dropout'],
              fusion_units=hp['fusion_dense_units'], run_name=name)
    results.append((name, hp, s['best_val_accuracy']))
for name, hp, val_acc in sorted(results, key=lambda r: -r[2]):
    print(f'{name}: val_acc={val_acc:.4f}  {hp}')

## 7. (Optional stretch) Stage-2 fine-tuning
Unfreeze the EfficientNet backbone and train at a low LR. Keep only if it helps.

In [ ]:
# s = train('fusion', learning_rate=1e-5, trainable_backbone=True, run_name='fusion_ft')
# evaluate('fusion', run_name='fusion_ft')

## 8. (Optional) Save artifacts to Google Drive
So trained models/figures survive the runtime disconnecting.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r artifacts /content/drive/MyDrive/DeepLearningProject_artifacts